In [0]:
# Read silver tables
transactions_silver = spark.table("jarvis_training.silver.transactions_data_silver")
users_silver = spark.table("jarvis_training.silver.users_data_silver")
cards_silver = spark.table("jarvis_training.silver.cards_data_silver")

In [0]:
# Creat the base gold table using the transactions silver table
from pyspark.sql.functions import *

fraud_transcation_gold_df = (
    transactions_silver
    .select(
        "transaction_id",
        "transaction_ts",
        "transaction_date",
        "transaction_year",
        "transaction_month",
        "transaction_day",
        "day_of_week_num",
        "day_of_week",
        "hour_of_day",
        "time_of_day",
        "client_id",
        "card_id",
        "amount",
        "use_chip",
        "merchant_id",
        "merchant_city",
        "merchant_state",
        "zip",
        "mcc_code",
        "mcc_description",
        "errors",
        "is_fraud"
    )
    .withColumn("week_start", to_date(date_trunc("week", col("transaction_ts"))))
    .withColumn("year_month", date_format(col("transaction_date"), "yyyy-MM"))
    .withColumn("abs_amount", abs(col("amount")))
    .withColumn("is_negative_amount", col("amount") < 0)
    .withColumn(
    "positive_amount",
    when(col("amount") > 0, col("amount")).otherwise(lit(0.0))
)
)

display(fraud_transcation_gold_df.limit(20))

fraud_transcation_gold_df.write.mode("overwrite").saveAsTable("jarvis_training.gold.fraud_transcation_gold")

transaction_id,transaction_ts,transaction_date,transaction_year,transaction_month,transaction_day,day_of_week_num,day_of_week,hour_of_day,time_of_day,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc_code,mcc_description,errors,is_fraud,week_start,year_month,abs_amount,is_negative_amount,positive_amount
7475745,2010-01-01T06:51:00.000Z,2010-01-01,2010,1,1,6,Friday,6,Morning,303,984,92.22,Swipe Transaction,35166,Windsor,CO,80550,5411,"Grocery Stores, Supermarkets",null,false,2009-12-28,2010-01,92.22,false,92.22
7475861,2010-01-01T07:17:00.000Z,2010-01-01,2010,1,1,6,Friday,7,Morning,203,5375,103.92,Swipe Transaction,81833,Islandton,SC,29929,5912,Drug Stores and Pharmacies,null,false,2009-12-28,2010-01,103.92,false,103.92
7476557,2010-01-01T10:04:00.000Z,2010-01-01,2010,1,1,6,Friday,10,Morning,119,5928,7.48,Swipe Transaction,87625,Las Vegas,NV,89146,5812,Eating Places and Restaurants,null,false,2009-12-28,2010-01,7.48,false,7.48
7478180,2010-01-01T16:00:00.000Z,2010-01-01,2010,1,1,6,Friday,16,Afternoon,1444,5965,18.07,Online Transaction,16798,ONLINE,null,null,4121,Taxicabs and Limousines,null,false,2009-12-28,2010-01,18.07,false,18.07
7478536,2010-01-01T17:42:00.000Z,2010-01-01,2010,1,1,6,Friday,17,Evening,154,5393,-62.0,Swipe Transaction,43293,Fayetteville,NC,28312,5499,Miscellaneous Food Stores,null,false,2009-12-28,2010-01,62.0,true,0.0
7478723,2010-01-01T18:46:00.000Z,2010-01-01,2010,1,1,6,Friday,18,Evening,866,2110,145.24,Swipe Transaction,60569,Pittsburgh,PA,15211,5300,Wholesale Clubs,null,false,2009-12-28,2010-01,145.24,false,145.24
7479097,2010-01-01T21:00:00.000Z,2010-01-01,2010,1,1,6,Friday,21,Night,1787,2228,70.51,Swipe Transaction,24339,Lexington,KY,40509,5812,Eating Places and Restaurants,null,false,2009-12-28,2010-01,70.51,false,70.51
7479828,2010-01-02T07:05:00.000Z,2010-01-02,2010,1,2,7,Saturday,7,Morning,585,5881,-65.0,Swipe Transaction,41260,Bastrop,TX,78602,5541,Service Stations,null,false,2009-12-28,2010-01,65.0,true,0.0
7480914,2010-01-02T11:51:00.000Z,2010-01-02,2010,1,2,7,Saturday,11,Morning,712,5506,56.93,Swipe Transaction,26810,Chula Vista,CA,91910,5541,Service Stations,null,false,2009-12-28,2010-01,56.93,false,56.93
7481114,2010-01-02T12:42:00.000Z,2010-01-02,2010,1,2,7,Saturday,12,Afternoon,868,3292,51.2,Swipe Transaction,60569,Oklahoma City,OK,73127,5300,Wholesale Clubs,null,false,2009-12-28,2010-01,51.2,false,51.2


In [0]:
# Create the gold table for fraud daily summary
fraud_daily_summary_gold_df = (
    fraud_transcation_gold_df
    .groupBy("transaction_date")
    .agg(
        count("*").alias("total_transcation_count"),
        countDistinct("client_id").alias("unique_users"),
        countDistinct("merchant_id").alias("unique_merchants"),
        sum(when(col("is_fraud") == True, 1).otherwise(0)).alias("fraud_transcation_count"),
        countDistinct(when(col("is_fraud") == True, col("client_id"))).alias("unique_fraud_users"),
        round(sum("amount"), 2).alias("total_amount"),
        round(
            sum(when(col("is_fraud") == True, col("abs_amount")).otherwise(lit(0.0))),
            2
        ).alias("fraud_amount"),
        round(
            sum(when(col("is_fraud") == True, 1).otherwise(0)) / count("*"),
            4
        ).alias("fraud_rate"),
        coalesce(
            round(avg(when(col("is_fraud") == True, col("abs_amount"))), 2),
            lit(0.0)
            ).alias("avg_fraud_amount"),
        coalesce(
            round(avg(when(col("is_fraud") == False, col("abs_amount"))), 2),
            lit(0.0)
            ).alias("avg_non_fraud_amount")
    )
    .orderBy("transaction_date")
)

display(fraud_daily_summary_gold_df.limit(20))

fraud_daily_summary_gold_df.write.mode("overwrite").saveAsTable("jarvis_training.gold.fraud_daily_summary_gold")

transaction_date,total_transcation_count,unique_users,unique_merchants,fraud_transcation_count,unique_fraud_users,total_amount,fraud_amount,fraud_rate,avg_fraud_amount,avg_non_fraud_amount
2010-01-01,3463,978,1010,1,1,124498.32,0.19,3.0E-4,0.19,51.49
2010-01-02,2989,954,986,0,0,138700.62,0.0,0.0,0.0,54.46
2010-01-03,3311,983,1009,1,1,135016.77,339.0,3.0E-4,339.0,49.12
2010-01-04,3244,988,958,2,1,131315.75,11.64,6.0E-4,5.82,48.96
2010-01-05,3330,980,995,1,1,143760.66,8.76,3.0E-4,8.76,52.86
2010-01-06,3365,992,1034,0,0,139046.49,0.0,0.0,0.0,51.33
2010-01-07,3346,984,1014,2,1,150784.43,629.54,6.0E-4,314.77,55.45
2010-01-08,3016,973,994,4,4,142249.31,383.24,0.0013,95.81,54.94
2010-01-09,3102,956,1018,1,1,138220.61,23.1,3.0E-4,23.1,61.07
2010-01-10,3416,988,1049,5,3,150998.46,530.01,0.0015,106.0,53.83


In [0]:
# Create the gold table for merchant summary
fraud_merchant_gold_df = (
    fraud_transcation_gold_df
    .withColumn(
        "amount_bucket",
        when(col("abs_amount") < 50, "low_value")
         .when((col("abs_amount") >= 50) & (col("abs_amount") < 200), "medium_value")
         .when((col("abs_amount") >= 200) & (col("abs_amount") < 500), "high_value")
         .otherwise("very_high_value")
    )
    .select(
        "transaction_id",
        "transaction_ts",
        "transaction_date",
        "transaction_year",
        "transaction_month",
        "transaction_day",
        "week_start",
        "year_month",
        "day_of_week_num",
        "day_of_week",
        "hour_of_day",
        "time_of_day",
        "client_id",
        "merchant_id",
        "merchant_city",
        "merchant_state",
        "mcc_code",
        "mcc_description",
        "amount",
        "abs_amount",
        "is_negative_amount",
        "amount_bucket",
        "is_fraud"
    )
)

display(fraud_merchant_gold_df.limit(20))

fraud_merchant_gold_df.write.mode("overwrite").saveAsTable("jarvis_training.gold.fraud_merchant_gold")

transaction_id,transaction_ts,transaction_date,transaction_year,transaction_month,transaction_day,week_start,year_month,day_of_week_num,day_of_week,hour_of_day,time_of_day,client_id,merchant_id,merchant_city,merchant_state,mcc_code,mcc_description,amount,abs_amount,is_negative_amount,amount_bucket,is_fraud
7476366,2010-01-01T09:21:00.000Z,2010-01-01,2010,1,1,2009-12-28,2010-01,6,Friday,9,Morning,1759,17812,Oceanside,CA,5411,"Grocery Stores, Supermarkets",2.23,2.23,false,low_value,false
7476607,2010-01-01T10:15:00.000Z,2010-01-01,2010,1,1,2009-12-28,2010-01,6,Friday,10,Morning,1300,74934,Oakland,CA,3596,Miscellaneous Machinery and Parts Manufacturing,170.43,170.43,false,medium_value,false
7477477,2010-01-01T13:08:00.000Z,2010-01-01,2010,1,1,2009-12-28,2010-01,6,Friday,13,Afternoon,1452,50867,Houston,TX,5541,Service Stations,-87.0,87.0,true,medium_value,false
7479110,2010-01-01T21:05:00.000Z,2010-01-01,2010,1,1,2009-12-28,2010-01,6,Friday,21,Night,1529,88646,Pompano Beach,FL,5812,Eating Places and Restaurants,63.97,63.97,false,medium_value,false
7479183,2010-01-01T21:35:00.000Z,2010-01-01,2010,1,1,2009-12-28,2010-01,6,Friday,21,Night,217,44921,Summit Argo,IL,5812,Eating Places and Restaurants,21.49,21.49,false,low_value,false
7480393,2010-01-02T09:40:00.000Z,2010-01-02,2010,1,2,2009-12-28,2010-01,7,Saturday,9,Morning,737,46474,Medford,OR,7538,Automotive Service Shops,65.73,65.73,false,medium_value,false
7480921,2010-01-02T11:52:00.000Z,2010-01-02,2010,1,2,2009-12-28,2010-01,7,Saturday,11,Morning,712,26810,Chula Vista,CA,5541,Service Stations,-53.0,53.0,true,medium_value,false
7482329,2010-01-02T17:53:00.000Z,2010-01-02,2010,1,2,2009-12-28,2010-01,7,Saturday,17,Evening,260,14451,Mission,TX,5912,Drug Stores and Pharmacies,48.02,48.02,false,low_value,false
7482862,2010-01-02T22:05:00.000Z,2010-01-02,2010,1,2,2009-12-28,2010-01,7,Saturday,22,Night,194,22204,Fountain City,IN,5541,Service Stations,31.79,31.79,false,low_value,false
7482887,2010-01-02T22:17:00.000Z,2010-01-02,2010,1,2,2009-12-28,2010-01,7,Saturday,22,Night,194,22204,Fountain City,IN,5541,Service Stations,100.0,100.0,false,medium_value,false


In [0]:
# Create the gold table for user summary
users_base_df = (
    users_silver
    .select(
        "client_id",
        "gender",
        "current_age",
        "credit_score",
        "yearly_income",
        "total_debt",
        "num_credit_cards"
    )
)

first_fraud_df = (
    fraud_transcation_gold_df
    .filter(col("is_fraud") == True)
    .groupBy("client_id")
    .agg(min("transaction_date").alias("first_fraud_date"))
)

fraud_user_gold_df = (
    fraud_transcation_gold_df.alias("t")
    .join(users_base_df.alias("u"), on="client_id", how="left")
    .join(first_fraud_df.alias("f"), on="client_id", how="left")
    .withColumn(
        "before_after_flag",
        when(
            col("first_fraud_date").isNull(),
            lit("no_fraud_history")
        ).when(
            col("transaction_date") < col("first_fraud_date"),
            lit("before_fraud")
        ).when(
            col("transaction_date") > col("first_fraud_date"),
            lit("after_fraud")
        ).otherwise(lit("first_fraud_day"))
    )
    .select(
        "transaction_id",
        "client_id",
        "transaction_date",
        "week_start",
        "merchant_id",
        "amount",
        "abs_amount",
        "is_fraud",
        "gender",
        "current_age",
        "credit_score",
        "yearly_income",
        "total_debt",
        "num_credit_cards",
        "first_fraud_date",
        "before_after_flag"
    )
)

display(fraud_user_gold_df.limit(20))

fraud_user_gold_df.write.mode("overwrite").saveAsTable("jarvis_training.gold.fraud_user_gold")

transaction_id,client_id,transaction_date,week_start,merchant_id,amount,abs_amount,is_fraud,gender,current_age,credit_score,yearly_income,total_debt,num_credit_cards,first_fraud_date,before_after_flag
7475745,303,2010-01-01,2009-12-28,35166,92.22,92.22,false,Male,94,690,60080.0,1807.0,6,2013-02-16,before_fraud
7475861,203,2010-01-01,2009-12-28,81833,103.92,103.92,false,Female,58,776,30305.0,23392.0,3,2015-04-03,before_fraud
7476557,119,2010-01-01,2009-12-28,87625,7.48,7.48,false,Female,44,624,56605.0,87970.0,3,2016-06-19,before_fraud
7478180,1444,2010-01-01,2009-12-28,16798,18.07,18.07,false,Male,64,698,66885.0,144703.0,7,2014-04-07,before_fraud
7478536,154,2010-01-01,2009-12-28,43293,-62.0,62.0,false,Male,48,666,37468.0,73448.0,3,2010-08-06,before_fraud
7478723,866,2010-01-01,2009-12-28,60569,145.24,145.24,false,Male,34,639,61904.0,125358.0,3,2018-12-24,before_fraud
7479097,1787,2010-01-01,2009-12-28,24339,70.51,70.51,false,Female,52,689,58278.0,51539.0,6,2010-05-09,before_fraud
7479828,585,2010-01-02,2009-12-28,41260,-65.0,65.0,false,Male,52,584,41483.0,64715.0,1,2013-10-25,before_fraud
7480914,712,2010-01-02,2009-12-28,26810,56.93,56.93,false,Female,74,720,29130.0,8746.0,5,2013-10-19,before_fraud
7481114,868,2010-01-02,2009-12-28,60569,51.2,51.2,false,Male,78,706,31046.0,11436.0,6,2013-07-29,before_fraud
